In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
import pandas as pd

from iot_security_mlops.config_loader import load_config
from iot_security_mlops.data.load_data import drop_cols
from iot_security_mlops.utils_core import find_repo_root

# Model Training

The purpose of this notebook is to train a random forest model from sklearn using the default configuration on `training.csv` to try and replicate the performance described in the paper.

In [2]:
config_path = find_repo_root() / 'config.yaml'
config = load_config(config_path)

data_file = find_repo_root() / config.paths.train_data
df = pd.read_csv(data_file)
df = drop_cols(df)

# encode categorical columns
df = df.astype('category')
cat_columns = df.select_dtypes(['category']).columns
df[cat_columns] = df[cat_columns].apply(lambda z: z.cat.codes)

# split into train and test
x = df.iloc[:, :-1]
y = df.iloc[:, -1]
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2,
                                                    random_state=config.train.random_state)

# fit default random forest model
classifier = RandomForestClassifier(random_state=config.train.random_state)
classifier.fit(x_train, y_train)

# make predictions
y_pred = classifier.predict(x_test)

# output metrics
print("Random Forest, accuracy: " + str(metrics.accuracy_score(y_test, y_pred)) + " F1 score:" + str(metrics.f1_score(y_test, y_pred,average='weighted')))
cm = metrics.confusion_matrix(y_test,y_pred)
print(cm)

Random Forest, accuracy: 0.9832261138908744 F1 score:0.9818422958085257
[[  118     0    13     9    32    21]
 [    0   110    94     0     0     2]
 [    0     0 19966     2     1     0]
 [    0     0    45   148     1     0]
 [   35     0    12     0   129    21]
 [    2     0    51     1    10   162]]


The paper a test accuracy of 0.994 and an F1 score of 0.994. The slightly worse performance could be explained by the differences in how the input datasets were constructed.